# State of Data Brazil — Pipeline de Dados (Camada Bronze, PySpark)

**Tech Challenge — Ingestão de dados brutos**

Versão em **PySpark** da ingestão da camada Bronze — mesma lógica já validada na
versão em pandas, reescrita para usar Spark (exigência do desafio) e para ficar
próxima do que roda dentro de um **AWS Glue Job** (que também é Spark por baixo).

**Escopo da camada Bronze:**
- Carregar os 3 CSVs originais (2023/2024/2025) exatamente como recebidos — nenhum
  valor é filtrado, transformado ou descartado.
- **Sanitizar os nomes das colunas** (não os valores): os cabeçalhos brutos da
  pesquisa usam pontos, barras, espaços, acentos e interrogações
  (ex: `1.f.8_Relação com outras pessoas da empresa, em momentos de trabalho`) —
  isso quebra o Spark (`.` tem significado especial em expressões de coluna) e
  **não é aceito pelo Glue Data Catalog / Athena**. A sanitização usa
  `src/pipeline_utils.sanitize_colname`, que mantém o código da pergunta como
  prefixo (garante unicidade) e troca tudo que não é `[a-z0-9_]` por `_`.
- Registrar metadados de ingestão (ano, arquivo de origem, timestamp).
- Persistir em **Parquet particionado por ano**, no mesmo layout de chave que
  será usado no S3 (`s3://<bucket>/bronze/ano=<ano>/`).


In [1]:
import sys
from pathlib import Path
from datetime import datetime, timezone

sys.path.insert(0, str(Path('..').resolve() / 'src'))
from pipeline_utils import find_project_root, sanitize_colname, dedupe_names, get_spark_session

from pyspark.sql import functions as F

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
BRONZE_DIR = PROJECT_ROOT / 'data' / 'bronze'
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", PROJECT_ROOT)

spark = get_spark_session('state-of-data-bronze')
spark


Raiz do projeto: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1


c:\Users\fhca02\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## 1. Carregamento dos arquivos brutos

Lemos cada CSV com todas as colunas como `string` (equivalente ao `dtype=str` do pandas), preservando os valores exatamente como foram coletados — tipagem correta é responsabilidade da Silver.

In [2]:
RAW_FILES = {
    2023: RAW_DIR / '2023.csv',
    2024: RAW_DIR / '2024.csv',
    2025: RAW_DIR / '2025.csv',
}

dfs_bronze = {}

for ano, path in RAW_FILES.items():
    df = (
        spark.read
        .option('header', True)
        .option('multiLine', True)
        .option('escape', '"')
        .option('encoding', 'UTF-8')
        .csv(str(path))
    )
    dfs_bronze[ano] = df
    print(f"{ano}: {df.count():,} linhas x {len(df.columns)} colunas".replace(',', '.'))


2023: 5.293 linhas x 399 colunas
2024: 5.217 linhas x 403 colunas
2025: 3.495 linhas x 388 colunas


## 2. Sanitização dos nomes de coluna

Renomeação **posicional** (`toDF`) — evita que o Spark tente interpretar os pontos do nome bruto como acesso a campo aninhado. O mapeamento nome bruto → nome saneado é salvo junto (útil para auditoria e para debugar no Glue Catalog depois).

In [3]:
column_maps = {}

for ano, df in dfs_bronze.items():
    original_cols = df.columns
    sanitized = dedupe_names([sanitize_colname(c) for c in original_cols])
    dfs_bronze[ano] = df.toDF(*sanitized)
    column_maps[ano] = list(zip(original_cols, sanitized))
    print(f"{ano}: exemplo de renomeacao -> {column_maps[ano][:3]}")


2023: exemplo de renomeacao -> [("('P0', 'id')", 'p0_id'), ("('P1_a ', 'Idade')", 'p1_a_idade'), ("('P1_a_1 ', 'Faixa idade')", 'p1_a_1_faixa_idade')]
2024: exemplo de renomeacao -> [('0.a_token', 'c_0_a_token'), ('0.d_data/hora_envio', 'c_0_d_data_hora_envio'), ('1.a_idade', 'c_1_a_idade')]
2025: exemplo de renomeacao -> [('0.a_token', 'c_0_a_token'), ('0.d_data/hora_envio', 'c_0_d_data_hora_envio'), ('1.a_idade', 'c_1_a_idade')]


In [4]:
import pandas as pd

DICT_DIR = PROJECT_ROOT / 'data' / 'dictionary'
DICT_DIR.mkdir(parents=True, exist_ok=True)

for ano, mapping in column_maps.items():
    pd.DataFrame(mapping, columns=['coluna_bruta', 'coluna_saneada']) \
        .to_csv(DICT_DIR / f'bronze_column_mapping_{ano}.csv', index=False)

print("Mapas de coluna salvos em data/dictionary/bronze_column_mapping_<ano>.csv")


Mapas de coluna salvos em data/dictionary/bronze_column_mapping_<ano>.csv


## 3. Perfil rápido de cada base

In [5]:
for ano, df in dfs_bronze.items():
    total_cells = df.count() * len(df.columns)
    null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
    total_nulls = sum(null_counts.first().asDict().values())
    pct = (total_nulls / total_cells) * 100
    print(f"{ano}: {df.count():,} linhas x {len(df.columns)} colunas | % nulos: {pct:.1f}%".replace(',', '.'))


2023: 5.293 linhas x 399 colunas | % nulos: 56.7%
2024: 5.217 linhas x 403 colunas | % nulos: 57.2%
2025: 3.495 linhas x 388 colunas | % nulos: 61.9%


## 4. Metadados de ingestão (lineage)

In [6]:
ingestion_ts = datetime.now(timezone.utc).isoformat()

for ano, df in dfs_bronze.items():
    dfs_bronze[ano] = (
        df
        .withColumn('_ano_pesquisa', F.lit(ano))
        .withColumn('_arquivo_origem', F.lit(RAW_FILES[ano].name))
        .withColumn('_data_ingestao', F.lit(ingestion_ts))
    )

dfs_bronze[2025].select('_ano_pesquisa', '_arquivo_origem', '_data_ingestao').show(3, truncate=False)


+-------------+---------------+--------------------------------+
|_ano_pesquisa|_arquivo_origem|_data_ingestao                  |
+-------------+---------------+--------------------------------+
|2025         |2025.csv       |2026-09-02T04:29:26.495888+00:00|
|2025         |2025.csv       |2026-09-02T04:29:26.495888+00:00|
|2025         |2025.csv       |2026-09-02T04:29:26.495888+00:00|
+-------------+---------------+--------------------------------+
only showing top 3 rows


## 5. Persistência da camada Bronze (Parquet particionado por ano)

In [7]:
for ano, df in dfs_bronze.items():
    out_path = BRONZE_DIR / f'ano={ano}'
    df.write.mode('overwrite').parquet(str(out_path))
    print(f"Salvo: {out_path}")


Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1\data\bronze\ano=2023
Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1\data\bronze\ano=2024
Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1\data\bronze\ano=2025


## 6. Checagem de sanidade pós-gravação

In [8]:
for ano in RAW_FILES:
    df_check = spark.read.parquet(str(BRONZE_DIR / f'ano={ano}'))
    original_rows = dfs_bronze[ano].count()
    status = 'OK' if df_check.count() == original_rows else 'DIVERGENTE'
    print(f"{ano}: {df_check.count():,} linhas lidas | original {original_rows:,} | {status}".replace(',', '.'))


2023: 5.293 linhas lidas | original 5.293 | OK
2024: 5.217 linhas lidas | original 5.217 | OK
2025: 3.495 linhas lidas | original 3.495 | OK


## 7. Próximos passos

- **Silver:** unificar os 3 anos usando `data/dictionary/mapa_campos.csv` (já
  atualizado para referenciar os nomes de coluna saneados desta Bronze).
- **AWS:** este mesmo código roda como AWS Glue Job quase sem alteração — troca
  `SparkSession.builder.master('local[*]')` por `GlueContext`/`glueContext.spark_session`
  e os paths locais por `s3://<bucket>/...`.
